# Négatifs appariés : que contiennent-ils ?

Un négatif apparié est *présumé* : ni annoté positif, ni écouté. Ce notebook mesure ce qu'on tire vraiment, stratégie par stratégie (`benchmark.pairing`), et prépare une feuille d'écoute.

- **Lecture seule**, sauf la feuille d'écoute, écrite dans `data/reports/` (sur C:, ignoré par git).
- **Ne pas committer les sorties** (lecteurs audio : données de l'ONF et de Biophonia).
- Tout se règle dans la cellule **Réglages**, puis *Run All*.

In [ ]:
import copy
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from blanci import explore as ex
from blanci import explore_plots as ep
from blanci.baselines import evaluation_windows, window_features
from blanci.config import config_path, load_config
from blanci.evaluate import wilson_interval

# Racine du dépôt : les chemins de la config y sont relatifs (le notebook tourne dans notebooks/).
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists())
os.chdir(ROOT)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 120)
pd.set_option("display.width", 200)

## Réglages

In [ ]:
CONFIG = ROOT / "config" / "local.yaml"
STRATEGIES = ["nearest", "same_day", "other_day"]
N_SPECTRE = 150  # fenêtres lues par sorte de négatif pour l'indice spectral
ANALYSE_DISTANCE = True  # contraste des fenêtres non annotées selon la distance (≈ 4 min)
ECOUTE_STRATEGIE = "nearest"
ECOUTE_SORTE = "same_recording"  # same_recording, same_day ou other_day
N_ECOUTE = 20
GRAINE = 0

cfg = load_config(CONFIG if CONFIG.exists() else None)
con = ex.open_readonly(config_path(cfg, "db"))
DETECTEURS = ex.detector_models(con)
print("détecteurs importés :", DETECTEURS or "aucun")

## 1. Ce que couvrent les annotations

Pour chaque enregistrement à annotation positive : fenêtres annotées, secondes couvertes, et détections BlanciNet hors de toute annotation positive — du chant probable, non annoté. L'expert a validé une partie des détections, il n'a pas marqué tout le chant : dans un enregistrement positif, « non annotée » ne veut pas dire « négative ».

In [ ]:
coverage = ex.annotation_coverage(con)
coverage["share_covered"] = coverage["covered_s"] / coverage["duration_s"]
columns = ["n_positive", "covered_s", "share_covered", "detections_outside", "median_score_outside"]
display(coverage[columns].describe().round(2))
coverage.round(2)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
axes[0].hist(coverage["share_covered"], bins=20)
axes[0].set_xlabel("part de l'enregistrement couverte par les annotations positives")
axes[0].set_ylabel("enregistrements")
axes[1].scatter(coverage["covered_s"], coverage["detections_outside"])
axes[1].set_xlabel("secondes annotées positives")
axes[1].set_ylabel("détections hors annotations")
fig.tight_layout()

## 2. Ce que tire chaque stratégie

Les fenêtres d'évaluation des baselines (`evaluation_windows`) pour chaque stratégie : les annotées et les négatifs appariés présumés. Pour chaque sorte de négatif (`pairing`) : part détectée par BlanciNet (le plus haut des scores qui couvrent la moitié de la fenêtre ; vide = rien détecté ≥ 0,1).

In [ ]:
drawn = {}
for strategy in STRATEGIES:
    variant = copy.deepcopy(cfg)
    variant["benchmark"]["pairing"] = strategy
    windows = evaluation_windows(con, variant)
    negatives = windows[windows["presumed"]].copy()
    drawn[strategy] = negatives.join(ex.detector_scores(con, negatives))
annotated = windows[windows["y"] == 1]

rows = []
for strategy, negatives in drawn.items():
    for kind, group in negatives.groupby("pairing"):
        row = {"stratégie": strategy, "sorte": kind, "négatifs": len(group)}
        for detector in DETECTEURS:
            score = group[detector] if detector in group else pd.Series(np.nan, index=group.index)
            row[f"{detector} ≥ 0,1"] = score.notna().mean()
            row[f"{detector} ≥ 0,5"] = (score >= 0.5).mean()
        rows.append(row)
pd.DataFrame(rows).round(2)

## 3. Un indice spectral : le contraste en bande

Contraste en bande (la meilleure baseline sans encodeur) de `N_SPECTRE` fenêtres par sorte, contre les fenêtres annotées positives. Des négatifs dont le contraste ressemble à celui des positives contiennent probablement du chant.

In [ ]:
def contrast(rows: pd.DataFrame) -> np.ndarray:
    out = []
    for row in rows.itertuples():
        segment, sr = ex.read_segment(cfg, row.path, row.offset_s, row.dur_s)
        out.append(window_features(segment, sr, cfg["signal"]).fixed["band_contrast"])
    return np.array(out)


def pick(rows: pd.DataFrame) -> pd.DataFrame:
    return rows.sample(min(N_SPECTRE, len(rows)), random_state=GRAINE)


samples = {"annotées positives": contrast(pick(annotated))}
for strategy, negatives in drawn.items():
    for kind, group in negatives.groupby("pairing"):
        samples[f"{strategy} / {kind}"] = contrast(pick(group))

fig, ax = plt.subplots(figsize=(12, 4))
ax.boxplot(list(samples.values()), tick_labels=list(samples), showfliers=False)
ax.set_ylabel("contraste en bande (dB)")
ax.tick_params(axis="x", rotation=30)
fig.tight_layout()
pd.DataFrame({k: pd.Series(v).describe() for k, v in samples.items()}).T.round(2)

### Selon la distance aux annotations

Toutes les fenêtres non annotées des enregistrements positifs, par distance à l'annotation positive la plus proche. Si le contraste décroît avec la distance jusqu'au niveau des négatifs d'un autre jour, le chant déborde des annotations d'autant.

In [ ]:
if ANALYSE_DISTANCE:
    overview = ex.recordings_overview(con, cfg)
    parts = []
    for rec in overview[overview["n_positive"] > 0].itertuples():
        wav, sr = ex.read_recording(cfg, rec.path)
        grid = ex.recording_windows(con, cfg, rec.recording_id)
        free = grid[~grid["overlaps_positive"] & grid["y"].isna()]
        values = ex.window_indices(wav, sr, free, cfg)
        parts.append(free[["recording_id", "distance_to_positive_s"]].join(values["band_contrast"]))
    by_distance = pd.concat(parts, ignore_index=True)
    bins = pd.cut(by_distance["distance_to_positive_s"], [-0.01, 6, 15, 30, 60, 120])
    display(
        by_distance.groupby(bins, observed=True)["band_contrast"].agg(["size", "median"]).round(2)
    )
    display(
        pd.Series({k: np.median(v) for k, v in samples.items()})
        .round(2)
        .to_frame("contraste médian")
    )

## 4. Écouter

`N_ECOUTE` négatifs tirés au hasard parmi ceux de `ECOUTE_STRATEGIE` / `ECOUTE_SORTE`. Écouter chacun, puis noter les verdicts dans la cellule suivante.

In [ ]:
pool = drawn[ECOUTE_STRATEGIE]
pool = pool[pool["pairing"] == ECOUTE_SORTE]
listening = pool.sample(min(N_ECOUTE, len(pool)), random_state=GRAINE).reset_index(drop=True)
segments = []
for i, row in listening.iterrows():
    segment, sr = ex.read_segment(cfg, row["path"], row["offset_s"], row["dur_s"])
    segments.append((f"n° {i}", segment))
    scores = ", ".join(f"{d} {row[d]:.2f}" for d in DETECTEURS if d in row and pd.notna(row[d]))
    print(f"n° {i} — {row['path']} à {row['offset_s']:g} s — {scores or 'rien détecté'}")
    display(ep.listen(segment, sr))
if segments:
    ep.spectrogram_grid(segments, sr, cfg)

In [ ]:
# Verdict par numéro : "chant" (A. blanci audible), "fond" (pas d'A. blanci) ou "doute".
VERDICTS = {}  # ex. {0: "fond", 1: "chant", 2: "doute"}

sheet = listening.assign(verdict=listening.index.map(VERDICTS))
k = int((sheet["verdict"] == "chant").sum())
n = int(sheet["verdict"].isin(["chant", "fond"]).sum())
if n:
    lo, hi = wilson_interval(k, n)
    print(f"{ECOUTE_STRATEGIE} / {ECOUTE_SORTE} : {k} chant(s) sur {n} négatifs tranchés")
    print(f"contamination {k / n:.0%} (IC 95 % {lo:.0%}–{hi:.0%})")
    out = config_path(cfg, "reports") / f"ecoute_negatifs_{ECOUTE_STRATEGIE}_{ECOUTE_SORTE}.csv"
    sheet.drop(columns=["label", "y", "presumed"], errors="ignore").to_csv(out, index=False)
    print("feuille d'écoute :", out)
else:
    print("Remplir VERDICTS après l'écoute.")